# Complejidad de Kolmogorov y compresión

Exploración empírica de la complejidad de Kolmogorov usando compresores
como proxies, y aplicación a clustering y distancia de información.

**Artículo relacionado:** `05/01-complejidad-de-kolmogorov.md`

In [ ]:
import zlib
import math
import random
import numpy as np
from itertools import combinations

## 1. Aproximación de K(x) mediante compresión

In [ ]:
def K(x):
    """Aproxima K(x) en bytes usando zlib."""
    if isinstance(x, str):
        x = x.encode()
    elif isinstance(x, (list, tuple)):
        x = bytes(x)
    return len(zlib.compress(x, level=9))


def K_joint(x, y):
    """Aproxima K(x,y) concatenando x e y."""
    if isinstance(x, str): x = x.encode()
    if isinstance(y, str): y = y.encode()
    return len(zlib.compress(x + y, level=9))


def NCD(x, y):
    """Normalized Compression Distance entre x e y."""
    kx = K(x)
    ky = K(y)
    kxy = K_joint(x, y)
    return (kxy - min(kx, ky)) / max(kx, ky)


# Verificar propiedades de NCD
texts = {
    'aaa': 'a' * 200,
    'abc': 'abc' * 67,
    'random1': ''.join(chr(random.randint(65,90)) for _ in range(200)),
    'random2': ''.join(chr(random.randint(65,90)) for _ in range(200)),
    'espanol': 'el perro come la carne y el gato bebe la leche todos los dias ',
    'ingles':  'the dog eats the meat and the cat drinks the milk every day again ',
    'ruso':    'собака ест мясо а кот пьет молоко каждый день в этом мире да',
}

names = list(texts.keys())
print("NCD entre pares de textos:")
print(f"{'Par':<30} {'NCD':<8} {'Interpretación'}")
print('-' * 65)
for a, b in [('aaa','abc'), ('aaa','espanol'), ('espanol','ingles'),
             ('espanol','ruso'), ('random1','random2'), ('aaa','aaa')]:
    ncd = NCD(texts[a], texts[b])
    if ncd < 0.3:
        interp = "muy similares"
    elif ncd < 0.6:
        interp = "relación moderada"
    else:
        interp = "muy diferentes"
    print(f"{a+' vs '+b:<30} {ncd:<8.4f} {interp}")

### Ejercicio 1.1
Calcula NCD entre las siguientes secuencias y explica el resultado:

In [ ]:
# Ejercicio: calcula y explica
seq_a = bytes([i % 4 for i in range(500)])      # patrón periódico
seq_b = bytes([i % 4 for i in range(100, 600)]) # mismo patrón, desplazado
seq_c = bytes([random.randint(0,255) for _ in range(500)])  # aleatorio

# Tu solución:
# ncd_ab = NCD(seq_a, seq_b)
# ncd_ac = NCD(seq_a, seq_c)
# ncd_bc = NCD(seq_b, seq_c)
# print(f"NCD(a,b) = {ncd_ab:.4f}")
# print(f"NCD(a,c) = {ncd_ac:.4f}")
# print(f"NCD(b,c) = {ncd_bc:.4f}")

# Predicción antes de ejecutar:
# - NCD(a,b) debería ser... (¿alto o bajo? ¿por qué?)
# - NCD(a,c) debería ser... 
# - NCD(b,c) debería ser...

print("TODO: implementa la comparación y explica el resultado")

## 2. MDL: Minimum Description Length

In [ ]:
def mdl_polynomial_fit(data_x, data_y, max_degree=5):
    """
    Selección de modelo por MDL para ajuste polinómico.
    
    Criterio MDL simplificado:
    - L(H) = grado_del_polinomio * bits_por_coeficiente
    - L(D|H) = suma de log2(error_cuadrático_por_punto)
    """
    results = []
    n = len(data_x)
    
    for degree in range(1, max_degree + 1):
        # Ajustar polinomio de grado 'degree'
        coeffs = np.polyfit(data_x, data_y, degree)
        y_pred = np.polyval(coeffs, data_x)
        residuals = data_y - y_pred
        mse = np.mean(residuals**2)
        
        # L(H): coste del modelo (bits para describir coeficientes)
        # Simplificado: cada coeficiente necesita ~16 bits de precisión
        bits_per_coeff = 16
        L_H = (degree + 1) * bits_per_coeff
        
        # L(D|H): coste de los datos dado el modelo
        # Approximación: -n/2 * log2(2*pi*e*mse) (fuente Gaussiana)
        if mse > 0:
            L_DH = n / 2 * math.log2(2 * math.pi * math.e * mse)
        else:
            L_DH = 0
        
        total = L_H + L_DH
        results.append({'degree': degree, 'L_H': L_H, 'L_DH': L_DH, 'total': total, 'mse': mse})
    
    return results


# Datos generados por un polinomio de grado 2 con ruido
random.seed(42)
np.random.seed(42)
x = np.linspace(-2, 2, 20)
y_true = 3*x**2 - x + 2  # Polinomio real: grado 2
y_noisy = y_true + np.random.normal(0, 0.5, len(x))  # Ruido gaussiano

results = mdl_polynomial_fit(x, y_noisy)

print("MDL para ajuste polinómico de datos con ruido (modelo real: grado 2):")
print(f"{'Grado':<8} {'L(H)':<10} {'L(D|H)':<12} {'Total MDL':<12} {'MSE':<12} {'MDL prefiere'}")
print('-' * 65)

min_mdl = min(r['total'] for r in results)
for r in results:
    marker = " ← ÓPTIMO" if abs(r['total'] - min_mdl) < 0.1 else ""
    print(f"{r['degree']:<8} {r['L_H']:<10.1f} {r['L_DH']:<12.1f} {r['total']:<12.1f} {r['mse']:<12.4f}{marker}")

print()
print("MDL equilibra complejidad del modelo (L_H) con calidad de ajuste (L_DH).")
print("El grado óptimo debería ser cercano al grado real (2).")

### Ejercicio 2.1
Repite el experimento MDL con datos generados por un **polinomio de grado 4** y ruido mayor (σ=2.0).
¿Cuándo MDL identifica correctamente el grado real? ¿Cuándo se equivoca?

In [ ]:
# Tu solución aquí:
# x = np.linspace(-2, 2, 30)
# y_true = x**4 - 2*x**2 + 1  # Polinomio real: grado 4
# ...

print("TODO: genera datos con polinomio grado 4 y analiza el MDL")

## 3. Distancia de información algorítmica

In [ ]:
def algorithmic_mutual_information(x, y):
    """
    Información mutua algorítmica aproximada:
    I(x:y) ≈ K(x) + K(y) - K(x,y)
    """
    kx = K(x)
    ky = K(y)
    kxy = K_joint(x, y)
    return kx + ky - kxy


# Verificar propiedades de la información mutua algorítmica
x1 = b'the quick brown fox jumps over the lazy dog ' * 5
y1 = b'the quick brown fox jumps over the lazy cat ' * 5  # similar a x1
z1 = bytes([random.randint(0, 127) for _ in range(len(x1))])  # no relacionado

print("Información mutua algorítmica (I(x:y) ≈ K(x) + K(y) - K(xy)):")
print()

pairs = [('texto vs similar', x1, y1), 
         ('texto vs aleatorio', x1, z1),
         ('aleatorio vs aleatorio', z1, bytes([random.randint(0,127) for _ in range(len(z1))]))]

for name, a, b in pairs:
    ka, kb = K(a), K(b)
    kab = K_joint(a, b)
    ami = ka + kb - kab
    print(f"{name}:")
    print(f"  K(x)={ka}, K(y)={kb}, K(xy)={kab}")
    print(f"  I(x:y) ≈ {ami} bytes")
    print()

In [ ]:
# === Tests automáticos ===
import zlib

# Test 1: K de cadena constante es mucho menor que K de cadena aleatoria
zeros_1000 = b'\x00' * 1000
random_1000 = bytes(range(256)) * 4  # 1024 bytes pseudo-estructurados
# El compresor debe reducir significativamente los ceros
k_zeros = len(zlib.compress(zeros_1000, level=9))
k_full = len(zeros_1000)
assert k_zeros < k_full / 3, f'Compresión ineficiente para ceros: {k_zeros} vs {k_full}'

# Test 2: K(x) ≤ |x| + c (la descripción trivial es copiar x)
for test_str in ['hello world', 'abababababab', '01010101']:
    encoded = test_str.encode()
    k_val = K(test_str)
    assert k_val <= len(encoded) + 20, f'K({test_str!r}) = {k_val} > |x| + 20 = {len(encoded) + 20}'

# Test 3: cadena periódica es más compresible que cadena aleatoria del mismo tamaño
periodic = 'ab' * 100  # 200 chars, muy compresible
import random; random.seed(42)
random_str = ''.join(random.choice('abcdefghij') for _ in range(200))
assert K(periodic) < K(random_str), (
    f'Periódica K={K(periodic)} no es menor que aleatoria K={K(random_str)}')

# Test 4: invarianza bajo descripción (concatenar dos copias no duplica K)
s = 'hello world test string'
k_s = K(s)
k_ss = K(s + s)
# K(ss) debe ser algo mayor que K(s) pero no duplicarse
assert k_ss < 2 * k_s, f'K(ss) = {k_ss} no es menor que 2*K(s) = {2*k_s}'

print('✓ Todos los tests de kolmogorov-y-compresion pasaron correctamente.')